In [1]:
import numpy as np
import pickle

save_file_name = 'morpho2d1.pkl'

In [2]:
def CutCircle(dists,angles):
    nC = len(dists)
    conts = np.zeros(nC)
    if np.min(dists)<=-1:
        return conts, 0.0, 0.0
    
    id_neighbor = np.where(dists<1)[0]
    nN = len(id_neighbor)
    if nN==0:
        return conts, np.pi, np.pi*2
    
    ds = dists[id_neighbor]
    ts = angles[id_neighbor]
    sg = np.sign(ds)
    dtheta = np.mod(ts[:,None]-ts[None,:]+np.pi,2*np.pi)-np.pi
    ang1 = np.arccos(ds)
    tanb = (ds[None,:]/ds[:,None]-np.cos(dtheta))/(np.sin(np.abs(dtheta))+np.eye(nN))
    b = np.arctan(tanb)+np.pi*(1-sg[:,None])/2
    a1 = (b-ang1[:,None])*(dtheta>0)+ang1[:,None]
    a2 = (b-ang1[:,None])*(dtheta<0)+ang1[:,None]
    c1 = np.min(a1,axis=1)*(sg+1)/2+np.max(a1,axis=1)*(1-sg)/2
    c2 = np.min(a2,axis=1)*(sg+1)/2+np.max(a2,axis=1)*(1-sg)/2
    c12 = np.clip(c1+c2,0,2*np.pi)
    h = np.maximum((np.tan(c1)+np.tan(c2))*sg,0)*ds*sg
    arc = np.maximum(np.pi*np.maximum(np.sum((1-sg)/2),1)*2-np.sum(c12),0)
    volume = (np.sum(h*ds)+arc)/2
    conts[id_neighbor] = h
    
    return conts, volume, arc

In [3]:
def AdjacencyMatrix(x,y,u):
    nC = len(x)
    dx = -x[:,None]+x[None,:]
    dy = -y[:,None]+y[None,:]
    du = u[:,None]**2-u[None,:]**2
    dq = np.sqrt(dx**2+dy**2)+np.diag(u*3)
    angles = np.arctan2(dy,dx)
    dists = (du+dq**2)/(2*dq*u[:,None])
    volumes = np.zeros(nC)
    adjacency = np.zeros((nC,nC))
    arcs = np.zeros(nC)
    for i in range(nC):
        conts, volume, arc = CutCircle(dists[i],angles[i])
        adjacency[i] = conts*u[i]
        volumes[i] = volume*u[i]*u[i]
        arcs[i] = arc*u[i]
    return adjacency, volumes, arcs

In [4]:
def Anisotropy(x,y,adjacency):
    dx = -x[:,None]+x[None,:]
    dy = -y[:,None]+y[None,:]
    ang = np.arctan2(dy,dx)
    ai1 = np.sum(adj*np.cos(ang)*np.cos(ang),axis=1)
    ai2 = np.sum(adj*np.cos(ang)*np.sin(ang),axis=1)
    ai3 = np.sum(adj*np.sin(ang)*np.sin(ang),axis=1)
    sg = (ai1-ai3)>0
    elongation = np.arctan2(2*ai2,(ai1-ai3))/2+sg*np.pi/2
    return elongation

In [5]:
def MechanicRule(M,x,y,u,v,adj,arcs,delta_x=0.1,delta_u=0.1,v_ref=15,t_ref=1):
    # delta_x and delta_u are step sizes
    # v_ref and t_ref are the reference volume and tension
    nC = len(x)
    adj0 = np.sign(adj)
    v0 = v_ref*(M[0,:]+1)
    t0 = t_ref*(M[1,:]+1)
    dv = v0-v
    tn = t0[:,None]+t0[None,:]
    
    dx = -x[:,None]+x[None,:]
    dy = -y[:,None]+y[None,:]
    dq = np.sqrt(dx**2+dy**2)+np.eye(nC)
    du = (-u[:,None]**2+u[None,:]**2)/dq**2
    mx1 = np.sum(dx/dq*(tn-dq)*adj0,axis=0)
    my1 = np.sum(dy/dq*(tn-dq)*adj0,axis=0)
    mx2 = np.sum(dx/dq*(dv[None,:]*(1-du)+dv[:,None]*(1+du))*adj,axis=0)
    my2 = np.sum(dy/dq*(dv[None,:]*(1-du)+dv[:,None]*(1+du))*adj,axis=0)
    mu1 = np.sum((dv[None,:]-dv[:,None])*adj/dq,axis=0)*u
    mu2 = arcs*dv
    
    x_new = x+(mx1/(t_ref**2)+mx2/(v_ref**2))*delta_x
    y_new = y+(my1/(t_ref**2)+my2/(v_ref**2))*delta_x
    u_new = u+(mu1+mu2)/(v_ref**2)*delta_u

    x_new = x_new+np.random.normal(0,0.01,size=nC)
    y_new = y_new+np.random.normal(0,0.01,size=nC)
    
    return x_new, y_new, u_new

In [6]:
def AutomataRule(M,adj,W,V,B,r,K=1.0,dt=0.05):
    fM = M/(K+M)
    dM = r[:,None]*M*(B[:,None]-M+W@fM+V@fM@adj)
    return M+dM*dt

In [7]:
def CellDivision(M,x,y,u,adj,threshold=1.0):
    div_cell = np.where(M[0]>threshold)[0]
    if len(div_cell)==0:
        return M,x,y,u
    el = Anisotropy(x,y,adj)
    for ii in div_cell:
        M[0,ii] = 1e-5
        x = np.append(x,x[ii]+np.cos(el[ii]))
        y = np.append(y,y[ii]+np.sin(el[ii]))
        u = np.append(u,u[ii])
        x[ii] = x[ii]-np.cos(el[ii])*0.1
        y[ii] = y[ii]-np.sin(el[ii])*0.1
        M = np.column_stack((M,M[:,ii]))
    return M,x,y,u

In [8]:
def CellInitial(cx,cy):
    nx = np.arange(cx)
    ny = np.arange(cy)
    nx = np.tile(nx,(cy,1))
    ny = np.tile(ny,(cx,1))
    nx = np.reshape(nx,cx*cy)
    ny = np.reshape(ny.T,cx*cy)
    nx = np.random.rand(cx*cy)+nx*5
    ny = np.random.rand(cx*cy)+ny*5
    nw = np.random.rand(cx*cy)+3
    return nx, ny, nw

In [9]:
x,y,u = CellInitial(2,2)

nC = len(x)
nM = 4
max_iter = 20000
x_list = [x.copy()]
y_list = [y.copy()]
u_list = [u.copy()]
M = np.ones((nM,nC))
M[0,:] = 1e-5
M_list = [M.copy()]
W = np.zeros((nM,nM))
V = np.zeros((nM,nM))
B = np.array([0,1.0,1.0,-0.1])
r = np.array([0.1,1,2,2])
W[3,2] = 0.5
W[0,3] = 6.0
V[2,1] = -0.14
V[2,2] = 0.09
V[3,3] = 0.07

count = 0
while count<max_iter and len(x)<100:
    adj,vol,arcs = AdjacencyMatrix(x,y,u)
    M = AutomataRule(M,adj,W,V,B,r)
    x,y,u = MechanicRule(M,x,y,u,vol,adj,arcs)
    M,x,y,u = CellDivision(M,x,y,u,adj)
    x_list.append(x.copy())
    y_list.append(y.copy())
    u_list.append(u.copy())
    M_list.append(M.copy())
    count += 1
    print(count,end='\r')

20000

In [10]:
with open(save_file_name,'wb') as pk:
    pickle.dump((x_list,y_list,u_list,M_list),pk)